# Salary Feature Influence Analysis

This notebook analyzes which features are most related to NBA salary value before committing to a salary forecasting model.

The primary target is `salary_cap_share`, not raw USD salary:

```text
salary_cap_share = salary_usd / salary_cap_usd
```

Two views are included:

- Descriptive same-season analysis: useful for understanding salary structure, but not sufficient for forecasting.
- Predictive prior-season analysis: uses previous salary and previous-season performance to predict the next salary season with lower leakage risk.


In [ ]:
%pip install -q pandas pyarrow scikit-learn lightgbm mlflow

## 1. Paths And Data Loading

This notebook is designed for Colab and reads the project data folder from Google Drive by default.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = Path('/content').exists()
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/nba-scout-assistant')
LOCAL_PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = DRIVE_PROJECT_DIR if IN_COLAB and DRIVE_PROJECT_DIR.exists() else LOCAL_PROJECT_DIR
DATA_DIR = Path(os.getenv('NBA_SCOUT_DATA_DIR', PROJECT_ROOT / 'data')).expanduser().resolve()
GOLD_DIR = DATA_DIR / 'gold'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'salary_feature_influence'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('GOLD_DIR:', GOLD_DIR)
print('REPORT_DIR:', REPORT_DIR)

In [ ]:
from __future__ import annotations

import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 180)

In [ ]:
SALARY_PATH = GOLD_DIR / 'salary_training_clean.parquet'
if not SALARY_PATH.exists():
    raise FileNotFoundError(f'Missing salary gold dataset: {SALARY_PATH}')

salary = pd.read_parquet(SALARY_PATH)
print('salary shape:', salary.shape)
display(salary.head())
display(pd.DataFrame({'dtype': salary.dtypes.astype(str), 'missing_pct': salary.isna().mean(), 'n_unique': salary.nunique(dropna=True)}).sort_values('missing_pct', ascending=False).head(30))

## 2. Modeling Frame Design

The predictive frame uses previous-season features to predict the current season salary cap share.

Core hypothesis:

```text
current salary value ~= previous salary value + previous production + current age / role context
```

This avoids the most obvious leakage from using same-season production to explain that same season's salary.

In [ ]:
TARGET = 'salary_cap_share'
USD_TARGET = 'salary_usd'

SPLIT_TRAIN_SEASONS = {'2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23'}
SPLIT_VALIDATION_SEASONS = {'2023-24'}
SPLIT_TEST_SEASONS = {'2024-25'}

PRIOR_LAG_COLUMNS = [
    'salary_usd',
    'salary_cap_share',
    'minutes',
    'usage_pct',
    'points_per_100',
    'assists_per_100',
    'rebounds_per_100',
    'true_shooting_pct',
    'three_point_attempt_rate',
    'free_throw_rate',
    'turnover_rate',
    'steal_rate',
    'block_rate',
    'defensive_rebound_rate',
    'foul_rate',
    'scoring_creation',
    'playmaking',
    'shooting',
    'rim_pressure',
    'rebounding',
    'perimeter_defense',
    'interior_defense',
    'two_way_impact',
]

CURRENT_CONTEXT_COLUMNS = [
    'age',
    'height',
    'weight',
    'position',
]

def assign_salary_split(season_label: object) -> str:
    """Input: season label. Output: temporal split for salary analysis."""
    if season_label in SPLIT_TRAIN_SEASONS:
        return 'train'
    if season_label in SPLIT_VALIDATION_SEASONS:
        return 'validation'
    if season_label in SPLIT_TEST_SEASONS:
        return 'test'
    return 'ignore'

def build_prior_season_salary_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Input: clean salary table. Output: current salary target with prior-season feature columns."""
    frame = df.copy()
    frame = frame.dropna(subset=['player_id', 'season_start_year', TARGET, USD_TARGET, 'salary_cap_usd']).copy()
    frame['player_id'] = pd.to_numeric(frame['player_id'], errors='coerce')
    frame['season_start_year'] = pd.to_numeric(frame['season_start_year'], errors='coerce')
    frame = frame.dropna(subset=['player_id', 'season_start_year']).copy()
    frame = frame.sort_values(['player_id', 'season_start_year']).reset_index(drop=True)

    for column in PRIOR_LAG_COLUMNS:
        if column in frame.columns:
            frame[f'prev_{column}'] = frame.groupby('player_id')[column].shift(1)

    frame['prev_season_start_year'] = frame.groupby('player_id')['season_start_year'].shift(1)
    frame['has_consecutive_prior_season'] = frame['prev_season_start_year'].eq(frame['season_start_year'] - 1)
    frame['split'] = frame['season_label'].map(assign_salary_split)
    frame = frame[frame['split'].isin(['train', 'validation', 'test'])].copy()
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors='coerce').clip(lower=0)
    frame[USD_TARGET] = pd.to_numeric(frame[USD_TARGET], errors='coerce')
    frame['salary_cap_usd'] = pd.to_numeric(frame['salary_cap_usd'], errors='coerce')
    frame = frame.dropna(subset=[TARGET, USD_TARGET, 'salary_cap_usd']).copy()
    return frame.reset_index(drop=True)

salary_modeling = build_prior_season_salary_frame(salary)
print('salary_modeling shape:', salary_modeling.shape)
display(salary_modeling.groupby(['season_label', 'split']).size().reset_index(name='rows'))
display(salary_modeling[['player_name', 'season_label', TARGET, 'prev_salary_cap_share', 'prev_minutes', 'prev_usage_pct', 'age', 'position']].head(10))

## 3. Feature Sets

The feature sets are intentionally compact first. If simple features explain salary well, the salary model should not be overcomplicated.

In [ ]:
def existing(columns: list[str], df: pd.DataFrame = salary_modeling) -> list[str]:
    """Input: candidate columns. Output: columns available in the modeling dataframe."""
    return [column for column in columns if column in df.columns]

FEATURE_SETS = {
    'previous_salary_only': existing([
        'prev_salary_cap_share',
        'prev_salary_usd',
    ]),
    'previous_salary_plus_age_role': existing([
        'prev_salary_cap_share',
        'prev_salary_usd',
        'age',
        'position',
        'height',
        'weight',
    ]),
    'prior_production_compact': existing([
        'prev_salary_cap_share',
        'age',
        'position',
        'prev_minutes',
        'prev_usage_pct',
        'prev_points_per_100',
        'prev_assists_per_100',
        'prev_rebounds_per_100',
        'prev_true_shooting_pct',
        'prev_scoring_creation',
        'prev_playmaking',
        'prev_rebounding',
        'prev_two_way_impact',
    ]),
    'prior_production_full': existing([
        'prev_salary_cap_share',
        'prev_salary_usd',
        'age',
        'height',
        'weight',
        'position',
        *[f'prev_{column}' for column in PRIOR_LAG_COLUMNS if column not in {'salary_usd', 'salary_cap_share'}],
    ]),
}

for name, features in FEATURE_SETS.items():
    print(name, len(features), features)

## 4. Descriptive Feature Influence

This section uses the train split only to avoid learning from validation/test seasons while deciding which features matter.

In [ ]:
def correlation_report(df: pd.DataFrame, features: list[str], target: str = TARGET) -> pd.DataFrame:
    """Input: dataframe, features, target. Output: numeric Pearson/Spearman correlation report."""
    rows = []
    train_df = df[df['split'].eq('train')].copy()
    for feature in features:
        if feature not in train_df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(train_df[feature]):
            continue
        pair = pd.concat([pd.to_numeric(train_df[feature], errors='coerce'), train_df[target]], axis=1).dropna()
        if len(pair) < 30:
            continue
        rows.append({
            'feature': feature,
            'rows': len(pair),
            'pearson': pair.iloc[:, 0].corr(pair.iloc[:, 1], method='pearson'),
            'spearman': pair.iloc[:, 0].corr(pair.iloc[:, 1], method='spearman'),
            'abs_pearson': abs(pair.iloc[:, 0].corr(pair.iloc[:, 1], method='pearson')),
            'abs_spearman': abs(pair.iloc[:, 0].corr(pair.iloc[:, 1], method='spearman')),
        })
    return pd.DataFrame(rows).sort_values(['abs_spearman', 'abs_pearson'], ascending=False).reset_index(drop=True)

analysis_features = sorted({feature for features in FEATURE_SETS.values() for feature in features})
correlation = correlation_report(salary_modeling, analysis_features)
display(correlation.head(30))
correlation.to_parquet(REPORT_DIR / 'salary_feature_correlation.parquet', index=False)

In [ ]:
def mutual_information_report(df: pd.DataFrame, features: list[str], target: str = TARGET) -> pd.DataFrame:
    """Input: dataframe, features, target. Output: mutual information feature ranking on train split."""
    train_df = df[df['split'].eq('train')].copy()
    selected = [feature for feature in features if feature in train_df.columns]
    numeric_features = [feature for feature in selected if pd.api.types.is_numeric_dtype(train_df[feature])]
    if not numeric_features:
        return pd.DataFrame(columns=['feature', 'mutual_information'])
    X = train_df[numeric_features].apply(pd.to_numeric, errors='coerce')
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    y = train_df[target].astype(float)
    scores = mutual_info_regression(X, y, random_state=42)
    return pd.DataFrame({'feature': numeric_features, 'mutual_information': scores}).sort_values('mutual_information', ascending=False).reset_index(drop=True)

mutual_information = mutual_information_report(salary_modeling, analysis_features)
display(mutual_information.head(30))
mutual_information.to_parquet(REPORT_DIR / 'salary_feature_mutual_information.parquet', index=False)

## 5. Candidate Models For Feature Importance

This section compares simple models before deeper salary modeling. The goal is to see whether compact prior-salary and prior-production features are enough.

In [ ]:
def get_one_hot_encoder() -> OneHotEncoder:
    """Input: none. Output: sklearn OneHotEncoder compatible across sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def build_preprocessor(df: pd.DataFrame, features: list[str], scale_numeric: bool = False) -> ColumnTransformer:
    """Input: dataframe and feature list. Output: sklearn preprocessor."""
    numeric_features = [feature for feature in features if pd.api.types.is_numeric_dtype(df[feature])]
    categorical_features = [feature for feature in features if feature not in numeric_features]
    numeric_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(numeric_steps), numeric_features),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='UNK')),
            ('onehot', get_one_hot_encoder()),
        ]), categorical_features),
    ])

def build_candidate_estimators() -> dict[str, tuple[object, bool]]:
    """Input: none. Output: candidate estimators and numeric scaling flag."""
    estimators = {
        'ridge': (Ridge(alpha=5.0), True),
        'elastic_net': (ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=20000, random_state=42), True),
        'random_forest': (RandomForestRegressor(n_estimators=400, min_samples_leaf=6, random_state=42, n_jobs=-1), False),
        'hist_gradient_boosting': (HistGradientBoostingRegressor(max_iter=250, learning_rate=0.04, l2_regularization=0.05, random_state=42), False),
    }
    if LIGHTGBM_AVAILABLE:
        estimators['lightgbm'] = (LGBMRegressor(n_estimators=350, learning_rate=0.03, num_leaves=15, min_child_samples=25, subsample=0.9, colsample_bytree=0.85, random_state=42, verbosity=-1), False)
    return estimators

In [ ]:
def evaluate_salary_model(model: Pipeline, df: pd.DataFrame, features: list[str], feature_set: str, model_name: str) -> list[dict[str, object]]:
    """Input: fitted model and modeling dataframe. Output: validation/test salary metrics."""
    rows = []
    for split in ['validation', 'test']:
        split_df = df[df['split'].eq(split)].copy()
        if split_df.empty:
            continue
        y_true = split_df[TARGET].astype(float)
        pred_share = pd.Series(model.predict(split_df[features]), index=split_df.index).clip(lower=0)
        pred_usd = pred_share * split_df['salary_cap_usd'].astype(float)
        rows.append({
            'feature_set': feature_set,
            'model': model_name,
            'split': split,
            'rows': len(split_df),
            'cap_share_mae': mean_absolute_error(y_true, pred_share),
            'cap_share_rmse': mean_squared_error(y_true, pred_share) ** 0.5,
            'cap_share_r2': r2_score(y_true, pred_share),
            'usd_mae': mean_absolute_error(split_df[USD_TARGET].astype(float), pred_usd),
        })
    return rows

def train_feature_set_models(df: pd.DataFrame) -> tuple[dict[str, Pipeline], pd.DataFrame]:
    """Input: salary modeling dataframe. Output: fitted models and evaluation table."""
    train_df = df[df['split'].eq('train')].copy()
    models = {}
    rows = []
    for feature_set, features in FEATURE_SETS.items():
        features = [feature for feature in features if feature in df.columns]
        if not features:
            continue
        for model_name, (estimator, scale_numeric) in build_candidate_estimators().items():
            model = Pipeline([
                ('preprocess', build_preprocessor(train_df, features, scale_numeric=scale_numeric)),
                ('model', estimator),
            ])
            model.fit(train_df[features], train_df[TARGET].astype(float))
            key = f'{feature_set}__{model_name}'
            models[key] = model
            rows.extend(evaluate_salary_model(model, df, features, feature_set, model_name))
    return models, pd.DataFrame(rows)

salary_models, salary_model_evaluation = train_feature_set_models(salary_modeling)
display(salary_model_evaluation.sort_values(['split', 'cap_share_mae']).head(50))
salary_model_evaluation.to_parquet(REPORT_DIR / 'salary_feature_set_model_evaluation.parquet', index=False)

## 6. Permutation Importance

Permutation importance measures how much validation error worsens when a feature is shuffled. This is usually more useful than raw tree impurity importance for mixed feature types.

In [ ]:
def permutation_importance_report(models: dict[str, Pipeline], evaluation: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """Input: fitted models and evaluation table. Output: validation permutation importance for the best model."""
    if evaluation.empty:
        return pd.DataFrame()
    validation_eval = evaluation[evaluation['split'].eq('validation')].sort_values('cap_share_mae')
    if validation_eval.empty:
        return pd.DataFrame()
    best = validation_eval.iloc[0]
    key = f"{best['feature_set']}__{best['model']}"
    model = models[key]
    features = FEATURE_SETS[str(best['feature_set'])]
    validation_df = df[df['split'].eq('validation')].copy()
    result = permutation_importance(
        model,
        validation_df[features],
        validation_df[TARGET].astype(float),
        scoring='neg_mean_absolute_error',
        n_repeats=20,
        random_state=42,
        n_jobs=-1,
    )
    importance = pd.DataFrame({
        'feature_set': best['feature_set'],
        'model': best['model'],
        'feature': features,
        'mae_increase_mean': result.importances_mean,
        'mae_increase_std': result.importances_std,
    }).sort_values('mae_increase_mean', ascending=False).reset_index(drop=True)
    return importance

permutation_importance_table = permutation_importance_report(salary_models, salary_model_evaluation, salary_modeling)
display(permutation_importance_table.head(30))
if not permutation_importance_table.empty:
    permutation_importance_table.to_parquet(REPORT_DIR / 'salary_permutation_importance.parquet', index=False)

## 7. Initial Modeling Takeaways

Use this final summary to decide whether the local salary model should be compact or more feature-rich.

In [ ]:
summary = {
    'rows': int(len(salary_modeling)),
    'train_rows': int(salary_modeling['split'].eq('train').sum()),
    'validation_rows': int(salary_modeling['split'].eq('validation').sum()),
    'test_rows': int(salary_modeling['split'].eq('test').sum()),
    'best_validation': salary_model_evaluation[salary_model_evaluation['split'].eq('validation')].sort_values('cap_share_mae').head(1).to_dict('records'),
    'top_correlation_features': correlation.head(10).to_dict('records'),
    'top_mutual_information_features': mutual_information.head(10).to_dict('records'),
    'top_permutation_features': permutation_importance_table.head(10).to_dict('records') if not permutation_importance_table.empty else [],
}

with open(REPORT_DIR / 'salary_feature_influence_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(json.dumps(summary, indent=2, default=str)[:4000])
print('Saved reports to:', REPORT_DIR)